In [ ]:
import sys
import cv2
import torch
import numpy as np
from pathlib import Path

# SSSS = Path(r"D:\Khanh\CardioVis\SSSS")
# sys.path.insert(0, str(SSSS))
from Models.DeepLabV3Plus.modeling import deeplabv3plus_resnet101

RAW_DIR = Path(r"D:\Khanh\CardioVis\raw")
OUTPUT_VIDEO = Path(r"D:\Khanh\CardioVis\raw_guideline_overlay.mp4")
CHECKPOINT = "checkpoints/cardio/cardio_run/fold1/best.pth"
IMG_SIZE = 224
NUM_CLASSES = 4
MAX_FRAMES = 1000
FPS = 18.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Class: 0=bg, 1=Epicardial adipose tissue, 2=Pericardium, 3=Phrenic nerve
CLASS_NAMES = ["Background", "Epicardial adipose tissue", "Pericardium", "Phrenic nerve"]
CLASS_COLORS_BGR = [
    (0, 0, 0), (0, 140, 255), (0, 255, 0), (0, 0, 255),
]
ALPHA = 0.5  # độ trong suốt cho 2 mask tô màu (class 1, 3)
PERICARDIUM_CLASS = 2
GUIDELINE_COLOR_BGR = (0, 255, 0)  # xanh lá (Pericardium = boundary + centerline)
DASH_LEN = 12
GAP_LEN = 8
LINE_THICKNESS = 2

ModuleNotFoundError: No module named 'cv2'

In [ ]:
model = deeplabv3plus_resnet101(num_classes=NUM_CLASSES, output_stride=8, pretrained_backbone=False)
state = torch.load(CHECKPOINT, map_location=DEVICE)
model.load_state_dict(state)
model = model.to(DEVICE)
model.eval()
print("Model loaded.")

In [ ]:
def preprocess_frame(frame):
    h, w = frame.shape[:2]
    img = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
    img = np.clip(img, 0, 255).astype(np.float32) / 255.0
    img = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    return img, (h, w)


def draw_dashed_line(img, pt1, pt2, color, thickness, dash_len, gap_len):
    x1, y1 = float(pt1[0]), float(pt1[1])
    x2, y2 = float(pt2[0]), float(pt2[1])
    length = np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
    if length < 1e-6:
        return
    step = dash_len + gap_len
    n = int(length / step) + 1
    for i in range(n):
        t0 = min(i * step / length, 1.0)
        t1 = min((i * step + dash_len) / length, 1.0)
        if t0 >= 1:
            break
        p0 = (int(x1 + t0 * (x2 - x1)), int(y1 + t0 * (y2 - y1)))
        p1 = (int(x1 + t1 * (x2 - x1)), int(y1 + t1 * (y2 - y1)))
        cv2.line(img, p0, p1, color, thickness)


def get_pericardium_centerline(mask, gap_threshold=5, smooth_win=3):
    """Lấy đường centerline trong lòng mask: mỗi cột x lấy y_min, y_max của mask rồi y_mid = (min+max)/2."""
    h, w = mask.shape[:2]
    points_by_x = []
    for x in range(w):
        col = mask[:, x]
        ys = np.where(col > 0)[0]
        if len(ys) > 0:
            y_mid = (ys.min() + ys.max()) / 2.0
            points_by_x.append((x, y_mid))
    if not points_by_x:
        return []
    points_by_x.sort(key=lambda p: p[0])
    if smooth_win >= 3 and len(points_by_x) >= smooth_win:
        xs = np.array([p[0] for p in points_by_x])
        ys = np.array([p[1] for p in points_by_x])
        kernel = np.ones(smooth_win) / smooth_win
        ys_smooth = np.convolve(ys, kernel, mode="same")
        points_by_x = list(zip(xs.tolist(), ys_smooth.tolist()))
    segments = []
    seg = [points_by_x[0]]
    for i in range(1, len(points_by_x)):
        if points_by_x[i][0] - points_by_x[i - 1][0] > gap_threshold:
            if len(seg) >= 2:
                segments.append(seg)
            seg = [points_by_x[i]]
        else:
            seg.append(points_by_x[i])
    if len(seg) >= 2:
        segments.append(seg)
    return segments


def draw_dashed_polyline(img, points, color, thickness, dash_len, gap_len, step_px=2):
    """Vẽ polyline (đường mở) dạng nét đứt."""
    if not points or len(points) < 2:
        return
    pts = np.array(points, dtype=np.float64)
    n = len(pts)
    samples = []
    d_total = 0.0
    for i in range(n - 1):
        p1, p2 = pts[i], pts[i + 1]
        seg_len = np.sqrt((p2[0] - p1[0]) ** 2 + (p2[1] - p1[1]) ** 2)
        if seg_len < 1e-6:
            continue
        num_steps = max(1, int(seg_len / step_px))
        for k in range(num_steps + 1):
            t = k / num_steps if num_steps > 0 else 1
            px = p1[0] + t * (p2[0] - p1[0])
            py = p1[1] + t * (p2[1] - p1[1])
            samples.append(((px, py), d_total))
            if k < num_steps:
                d_total += seg_len / num_steps
    if len(samples) < 2:
        return
    step = dash_len + gap_len
    in_dash = [((s[1] % step) < dash_len) for s in samples]
    for j in range(len(samples) - 1):
        if in_dash[j] and in_dash[j + 1]:
            p0 = (int(samples[j][0][0]), int(samples[j][0][1]))
            p1 = (int(samples[j + 1][0][0]), int(samples[j + 1][0][1]))
            cv2.line(img, p0, p1, color, thickness)


def draw_dashed_contour(img, contour, color, thickness, dash_len, gap_len, step_px=2):
    if contour is None or len(contour) < 2:
        return
    pts = contour.reshape(-1, 2).astype(np.float64)
    n = len(pts)
    samples = []
    d_total = 0.0
    for i in range(n):
        p1 = pts[i]
        p2 = pts[(i + 1) % n]
        seg_len = np.sqrt((p2[0] - p1[0]) ** 2 + (p2[1] - p1[1]) ** 2)
        if seg_len < 1e-6:
            continue
        num_steps = max(1, int(seg_len / step_px))
        for k in range(num_steps + 1):
            t = k / num_steps if num_steps > 0 else 1
            px = p1[0] + t * (p2[0] - p1[0])
            py = p1[1] + t * (p2[1] - p1[1])
            samples.append(((px, py), d_total))
            if k < num_steps:
                d_total += seg_len / num_steps
    if len(samples) < 2:
        return
    step = dash_len + gap_len
    in_dash = [((s[1] % step) < dash_len) for s in samples]
    for j in range(len(samples) - 1):
        if in_dash[j] and in_dash[j + 1]:
            p0 = (int(samples[j][0][0]), int(samples[j][0][1]))
            p1 = (int(samples[j + 1][0][0]), int(samples[j + 1][0][1]))
            cv2.line(img, p0, p1, color, thickness)


def overlay_guideline(frame, mask, h, w):
    """Overlay: class 1,3 tô màu như gốc; Pericardium = boundary nét đứt + centerline."""
    overlay = frame.copy()
    mask_resized = cv2.resize(
        mask.astype(np.uint8), (frame.shape[1], frame.shape[0]),
        interpolation=cv2.INTER_NEAREST
    )
    # Hai mask kia (Epicardial adipose tissue, Phrenic nerve): tô màu bán trong suốt như bản gốc
    for c in [1, 3]:
        color = CLASS_COLORS_BGR[c]
        overlay[mask_resized == c] = (
            (1 - ALPHA) * overlay[mask_resized == c].astype(np.float32) + ALPHA * np.array(color, dtype=np.float32)
        ).astype(np.uint8)
    # Pericardium (class 2): boundary nét đứt + centerline trong lòng mask
    pericardium = (mask_resized == PERICARDIUM_CLASS).astype(np.uint8)
    contours, _ = cv2.findContours(pericardium, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        if cv2.contourArea(cnt) < 50:
            continue
        draw_dashed_contour(overlay, cnt, GUIDELINE_COLOR_BGR, LINE_THICKNESS, DASH_LEN, GAP_LEN)
    centerline_segments = get_pericardium_centerline(pericardium, gap_threshold=5, smooth_win=5)
    for seg in centerline_segments:
        draw_dashed_polyline(overlay, seg, GUIDELINE_COLOR_BGR, LINE_THICKNESS, DASH_LEN, GAP_LEN)
    # Legend như bản gốc: 3 class
    box_x, box_y = 10, 10
    line_h = 26
    box_w, box_h = 380, 4 * line_h + 14
    cv2.rectangle(overlay, (box_x, box_y), (box_x + box_w, box_y + box_h), (40, 40, 40), -1)
    cv2.rectangle(overlay, (box_x, box_y), (box_x + box_w, box_y + box_h), (200, 200, 200), 1)
    for c in [1, 2, 3]:
        y = box_y + 8 + c * line_h
        cv2.rectangle(overlay, (box_x + 8, y - 14), (box_x + 30, y + 2), CLASS_COLORS_BGR[c], -1)
        label = "Pericardium (boundary+cut)" if c == 2 else CLASS_NAMES[c]
        cv2.putText(overlay, label, (box_x + 38, y), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
    return overlay.astype(np.uint8)

In [ ]:
all_paths = sorted(RAW_DIR.glob("frame_*.jpg"), key=lambda p: int(p.stem.split("_")[1]))
frame_paths = all_paths[:MAX_FRAMES]
print(f"Dùng {len(frame_paths)} frame đầu (tối đa {MAX_FRAMES}). Tổng trong raw: {len(all_paths)}")
if not frame_paths:
    raise SystemExit("Không có frame_*.jpg trong raw.")

In [ ]:
first = cv2.imread(str(frame_paths[0]))
if first is None:
    raise SystemExit(f"Không đọc được: {frame_paths[0]}")
h, w = first.shape[:2]
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, FPS, (w, h))
print(f"Output: {OUTPUT_VIDEO}, {w}x{h}, {FPS} fps")

In [ ]:
with torch.no_grad():
    for i, path in enumerate(frame_paths):
        frame = cv2.imread(str(path))
        if frame is None:
            continue
        img_t, (orig_h, orig_w) = preprocess_frame(frame)
        logits = model(img_t)
        pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        frame_overlay = overlay_guideline(frame, pred, orig_h, orig_w)
        out.write(frame_overlay)
        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(frame_paths)}")

out.release()
print("Done.", OUTPUT_VIDEO)